我们已经造好了零件，准备好了原材料，准备合体！（我来组成头部，我来组成。。。，烂梗王正是在下）




In [8]:
import torch
import torch.nn as nn
import numpy as np


def make_pad_mask(q, k, pad_idx = 0):
    # 承接上文，我们要构造一个mask，屏蔽src的padding部分

    # k.ne(pad_idx) -> 如果不是 pad 则是 True(1), 是 pad 则是 False(0)
    # 扩展维度以适应 Multi-Head Attention 的 score 矩阵形状
    # Score Shape: (Batch, Head, Q_Len, K_Len)

    # mask shape: (Batch, 1, 1, K_Len)
    # k.ne(pad_idx) — "not equal"，不等于 pad 的位置为 True
    mask = k.ne(pad_idx).unsqueeze(1).unsqueeze(2)  # (batch_size, 1, 1, src_len)
    # Score shape: (Batch, Head, Q_Len, K_Len), mask的shape 正好在在 head和Q_len这两个维度广播，做逐元素运算
    return mask

def make_subsequent_mask(seq_len):

    mask = torch.tril(torch.ones((seq_len, seq_len), dtype=torch.uint8))  # (seq_len, seq_len)
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

# 测试mask
q_sim = torch.tensor([[1, 2, 3]]) # batch size=1, seq_len=3  
k_sim= torch.tensor([[1, 2, 0,0]]) # batch size=1, seq_len=4, 其中0是pad_idx

pad_mask = make_pad_mask(q_sim, k_sim, pad_idx=0)   
print("Pad Mask Shape:", pad_mask.shape)  # (1, 1, 1, 4)
print("Pad Mask:", pad_mask.squeeze())  # 去掉多余的维度，查看实际mask内容

subsequent_mask = make_subsequent_mask(5)
print("\nSubsequent Mask Shape:", subsequent_mask.shape)  # (1, 1, 5, 5)
print("Subsequent Mask:", subsequent_mask.squeeze())


Pad Mask Shape: torch.Size([1, 1, 1, 4])
Pad Mask: tensor([ True,  True, False, False])

Subsequent Mask Shape: torch.Size([1, 1, 5, 5])
Subsequent Mask: tensor([[1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]], dtype=torch.uint8)


Encoder层就是堆叠多个EncoderLayer

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout = 0.1):
        super().__init__()

        # 这里应该是输入层，来自第一张
        # self.input_layer = TransformerEmbedding(vocab_size, d_model, dropout)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

    def forward(self, src, src_mask):
        # x = self.input_layer(src)  # (batch_size, src_len, d_model)

        for layer in self.layers:
            src = layer(src, src_mask)

        return self.norm(src)

堆叠 DecoderLayer作为Decoder层

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, dropout = 0.1):
        super().__init__()


        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        # 这里必须有一层投影层，将 d_model 映射到 vocab_size，这样才能找到对应的词表索引
        self.projection = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, enc_output, src_mask, tgt_mask):

        for layer in self.layers:
            # DecoderLayer 返回 (output, attn_map)，我们只保留最后一层的 attn_map
            tgt, attn_map = layer(tgt, enc_output, src_mask, tgt_mask)

        tgt = self.norm(tgt)

        return self.projection(tgt), attn_map

究极合体

举例说明代码中的combined_mask， 为什么突然多了一行这个呢？ 因为decoder需要同时满足两种mask， tgt的padding不应该看到，也不能看到未来的token，所以重叠两个矩阵，都是True的位置才是有效的。两个条件中任何一个说 "不能看"，就不能看。

```
tgt_mask (padding):          subsequent_mask (causal):
[[T, T, F, F]]               [[T, F, F, F],
                               [T, T, F, F],
 ↓ 广播到 (4,4)               [T, T, T, F],
[[T, T, F, F],                 [T, T, T, T]]
 [T, T, F, F],
 [T, T, F, F],
 [T, T, F, F]]

```
逐元素与操作，只有当两个位置都是True时，结果才是True
```
combined_mask:
[[T & T,  T & F,  F & F,  F & F],     [[T, F, F, F],
 [T & T,  T & T,  F & F,  F & F],  →   [T, T, F, F],
 [T & T,  T & T,  F & T,  F & F],      [T, T, F, F],  ← 注意这里！
 [T & T,  T & T,  F & T,  F & T]]      [T, T, F, F]]
```


为什么单独给推理阶段用？

这要从训练和推理的本质区别说起。
训练阶段：一把梭

训练时，目标序列 tgt 是完整已知的（因为有标注数据），所以模型的 forward 方法一次性跑完：
```
src → Encoder → enc_output
tgt（完整） → Decoder（一次性处理所有位置） → 输出
```
一次 forward 搞定，encoder 和 decoder 一起跑，没必要拆开。
推理阶段：逐词生成

推理时（比如翻译），你不知道完整的目标序列，要一个词一个词地生成：
```
第1步：输入 <BOS>           → 生成第1个词 "I"
第2步：输入 <BOS> I         → 生成第2个词 "love"
第3步：输入 <BOS> I love    → 生成第3个词 "you"
...
```
关键问题来了：每一步都要重新跑一次 Decoder，但 Encoder 的输出始终不变（因为源句子没变）。
如果不拆开，每生成一个词都要重新跑 encoder + decoder，等于 encoder 重复计算了 N 次（N = 目标序列长度），巨大浪费。
拆开之后，推理流程变成：
```
# 第1步：encoder 只跑一次，缓存结果
enc_output = model.encoder(src, src_mask)

# 第2步：decoder 循环跑 N 次，每次复用 enc_output
for step in range(max_len):
    output = model.decoder(tgt, enc_output, src_mask, tgt_mask)
    next_word = output.argmax(...)  # 取概率最大的词
    tgt = concat(tgt, next_word)    # 拼到目标序列后面
```
Encoder 跑 1 次，Decoder 跑 N 次。 这就是拆开的意义。


In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, n_heads, d_ff, n_layers, dropout = 0.1):
        super().__init__()


        self.src_embedding = TransformerInputLayer(src_vocab_size, d_model, dropout=dropout)
        self.tgt_embedding = TransformerInputLayer(tgt_vocab_size, d_model, dropout=dropout)

        self._encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers, dropout)
        self._decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, n_layers, dropout)

        self.src_pad_idx = 0
        self.tgt_pad_idx = 0


    def make_masks(self, src, tgt):
        src_mask = make_pad_mask(src, src, self.src_pad_idx)  # (batch_size, 1, 1, src_len)
        tgt_mask = make_pad_mask(tgt, tgt, self.tgt_pad_idx)  # (batch_size, 1, 1, tgt_len)
        subsequent_mask = make_subsequent_mask(tgt.size(1)).to(src.device)  # (1, 1, tgt_len, tgt_len)

        combined_mask = tgt_mask & subsequent_mask  # (batch_size, 1, tgt_len, tgt_len)

        return src_mask, combined_mask
    
    def forward(self, src, tgt):
        src_mask, tgt_mask = self.make_masks(src, tgt)

        enc_output = self._encoder(self.src_embedding(src), src_mask)
        dec_output, _ = self._decoder(self.tgt_embedding(tgt), enc_output, src_mask, tgt_mask)

        return dec_output
    
    # 专门暴露给推理阶段用的方法
    def encode(self, src, src_mask):
        return self._encoder(self.src_embedding(src), src_mask)
    
    def decode(self, tgt, enc_output, src_mask, tgt_mask):
        dec_output, _ = self._decoder(self.tgt_embedding(tgt), enc_output, src_mask, tgt_mask)
        return dec_output

来一个全流程测试，用随机数据跑一遍整个模型

In [ ]:
# --- 超参数 ---
src_vocab_size = 100
tgt_vocab_size = 100
d_model = 512
n_layers = 2  # 弄小点方便调试
n_head = 8

model = Transformer(src_vocab_size, tgt_vocab_size, d_model, n_head, d_ff=2048, n_layers=n_layers)
model.to(device)

# --- 伪造数据 ---

# Batch=2, Src_Len=5, Tgt_Len=6
src = torch.tensor([[1, 5, 6, 2, 0], [1, 9, 2, 0, 0]]).to(device)
tgt = torch.tensor([[1, 7, 3, 4, 8, 2], [1, 6, 8, 2, 0, 0]]).to(device)
# 注意：这里 0 是 pad，1 是 start，2 是 eos


print("Source Shape:", src.shape)  # (2, 5)
print("Target Shape:", tgt.shape)  # (2, 6)

output = model(src, tgt)

print("Output Shape:", output.shape)  # (2, 6, 100) -> (batch_size, tgt_len, vocab_size)


NameError: name 'TransformerEmbedding' is not defined

至此，你手中已经有了一个完整的transformer模型，你可以用它来做翻译等任务。
输入：源语言索引序列，目标语言索引序列
输出：每个位置下一个词的概率分布
